# Entraînement du Temporal Fusion Transformer

Ce notebook détaille toutes les étapes pour entraîner le modèle Temporal Fusion Transformer sur les jeux de données pickle présents dans le dossier `datasets/`.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


## Configuration Git & installation rapides

Les commandes ci-dessous reproduisent exactement les étapes demandées (passage dans `/content`, clonage du dépôt, installation editable, activation d’`autoreload`). Définissez éventuellement `TRADER_AUTO_REPO`, `TRADER_AUTO_BRANCH` ou `TRADER_AUTO_CLONE_DIR` pour personnaliser l’URL, la branche ou le dossier cible avant d’exécuter la cellule suivante.


In [ ]:
import os
import sys
from pathlib import Path
from urllib.parse import urlparse

repo_url = os.environ.get(
    "TRADER_AUTO_REPO", "https://github.com/clementremillieux/trader.git"
)
repo_branch = os.environ.get("TRADER_AUTO_BRANCH", "crypto_V3")
clone_dir = os.environ.get("TRADER_AUTO_CLONE_DIR")

parsed = urlparse(repo_url)
repo_name = Path(parsed.path).name or "repo"
if repo_name.endswith(".git"):
    repo_name = repo_name[:-4]
if not clone_dir:
    clone_dir = repo_name

print("Configuration utilisée:")
print(f"  URL     : {repo_url}")
print(f"  Branche : {repo_branch}")
print(f"  Dossier : {clone_dir}")

%cd /content
!rm -rf {clone_dir}
!git clone --branch {repo_branch} {repo_url} {clone_dir}
%cd /content/{clone_dir}

%cd /content/trader

%pip install -U \
  "pandas>=2.2.3,<3" "yfinance>=0.2.56,<0.3" "scikit-learn>=1.6.1,<2" \
  "python-dotenv>=1.1,<2" "numpy==1.26.4" "alpaca-py>=0.39.4,<1" \
  "matplotlib>=3.10.1,<4" "yahooquery>=2.3.7,<3" "httpx[http2]==0.28.1" \
  "python-binance>=1.0.28,<2" "binance-connector>=3.12.0,<4" \
  "pyarrow>=20,<21" "fastparquet>=2024.11,<2025" "tqdm>=4.66.5,<5"


PROJECT_ROOT = Path("/content") / clone_dir
os.environ["TRADER_AUTO_ROOT"] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"TRADER_AUTO_ROOT défini sur {PROJECT_ROOT}")


Configuration utilisée:
  URL     : https://github.com/clementremillieux/trader.git
  Branche : crypto_V3
  Dossier : trader
/content
Cloning into 'trader'...
remote: Enumerating objects: 1362, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 1362 (delta 76), reused 87 (delta 75), pack-reused 1271 (from 1)
Receiving objects: 100% (1362/1362), 19.72 MiB | 11.27 MiB/s, done.
Resolving deltas: 100% (554/554), done.
/content/trader
/content/trader
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 122.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 138.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 145.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.2/122.2 kB 13.7 MB/s eta 0:00:00
   ━━━━

In [ ]:
import json
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from scripts.train_tft import (
    TFTConfig,
    TemporalFusionTransformer,
    compute_statistics,
    estimate_num_samples,
    list_dataset_files,
    prepare_dataloader,
    setup_logging,
    train,
)


In [ ]:
training_params = {
    "train_prefix": "full_dataset_train_",
    "val_prefix": "full_dataset_val_",
    "max_train_files": None,  # Ajustez pour un entrainement rapide (ex: 5)
    "max_val_files": None,
    "limit_samples_per_file": None,  # Limiter le nb de séquences par fichier si besoin
    "stat_sample_fraction": 0.2,
    "batch_size": 64,
    "epochs": 15,
    "learning_rate": 1e-4,
    "grad_clip": 1.0,
    "lambda_reg": 0.1,
    "lambda_vol": 0.02,
    "seed": 42,
    "num_workers": 2,
    "mixed_precision": torch.cuda.is_available(),
    "save_every": 1,
}
output_dir = PROJECT_ROOT / "models" / "tft_notebook"
output_dir.mkdir(parents=True, exist_ok=True)
print(json.dumps(training_params, indent=2, ensure_ascii=False))
print(f"Les checkpoints seront sauvegardés dans: {output_dir}")


{
  "train_prefix": "full_dataset_focus_train_",
  "val_prefix": "full_dataset_focus_val_",
  "max_train_files": null,
  "max_val_files": null,
  "limit_samples_per_file": null,
  "stat_sample_fraction": 0.2,
  "batch_size": 64,
  "epochs": 15,
  "learning_rate": 0.0001,
  "grad_clip": 1.0,
  "lambda_reg": 0.1,
  "lambda_vol": 0.02,
  "seed": 42,
  "num_workers": 2,
  "mixed_precision": true,
  "save_every": 0
}
Les checkpoints seront sauvegardés dans: /content/trader/models/tft_notebook


In [ ]:
train_files = list_dataset_files(
    path="/content/drive/MyDrive/datatsets/full_dataset_focus",
    prefix=training_params["train_prefix"],
    max_files=training_params["max_train_files"],
)
val_files = list_dataset_files(
    path="/content/drive/MyDrive/datatsets/full_dataset_focus",
    prefix=training_params["val_prefix"],
    max_files=training_params["max_val_files"],
)
print(f"Fichiers train: {len(train_files)}")
print(f"Fichiers val: {len(val_files)}")
train_files[:3], val_files[:3]


FileNotFoundError: Aucun fichier trouvé pour le préfixe 'full_dataset_focus_train_'

In [ ]:
stats = compute_statistics(
    file_paths=train_files,
    sample_fraction=training_params["stat_sample_fraction"],
    seed=training_params["seed"],
    limit_samples_per_file=training_params["limit_samples_per_file"],
)
print(f"Dimension des features: {stats.feature_dim}")
print(f"Longueur de séquence: {stats.seq_len}")
print(f"Vocabulaire tau: {stats.tau_vocab_size}")
print("Poids de classes:", stats.class_weights)
print("Reg mean/std:", stats.reg_mean, stats.reg_std)
print("Vol mean/std:", stats.vol_mean, stats.vol_std)


In [ ]:
train_loader = prepare_dataloader(
    files=train_files,
    stats=stats,
    batch_size=training_params["batch_size"],
    shuffle_files=True,
    shuffle_samples=True,
    seed=training_params["seed"],
    limit_per_file=training_params["limit_samples_per_file"],
    num_workers=training_params["num_workers"],
    balance_classes=False,
)
val_loader = prepare_dataloader(
    files=val_files,
    stats=stats,
    batch_size=training_params["batch_size"],
    shuffle_files=False,
    shuffle_samples=False,
    seed=training_params["seed"],
    limit_per_file=training_params["limit_samples_per_file"],
    num_workers=training_params["num_workers"],
    balance_classes=False,
)

est_train = estimate_num_samples(train_files, stats)
est_val = estimate_num_samples(val_files, stats)
print(f"Séquences d'entraînement estimées: {est_train}")
print(f"Séquences de validation estimées: {est_val}")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = TFTConfig(
    input_dim=stats.feature_dim,
    seq_len=stats.seq_len,
    tau_vocab_size=stats.tau_vocab_size,
    hidden_dim=512,
    num_heads=8,
    num_transformer_blocks=6,
    dropout=0.2,
    conv_kernel_sizes=(3, 5, 7),
    conv_dilations=(1, 2, 4),
    static_dim=256,
)
model = TemporalFusionTransformer(cfg).to(device)
class_weights = torch.from_numpy(stats.class_weights)
setup_logging(verbose=True)
print(model)
print(f"Device utilisé: {device}")


In [ ]:
best_metrics = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    class_weights=class_weights,
    device=device,
    epochs=training_params["epochs"],
    lr=training_params["learning_rate"],
    grad_clip=training_params["grad_clip"],
    lambda_reg=training_params["lambda_reg"],
    lambda_vol=training_params["lambda_vol"],
    mixed_precision=training_params["mixed_precision"],
    output_dir=output_dir,
    save_every=training_params["save_every"],
)
best_metrics


In [ ]:
# Entraînement avec FocalLoss ajustée pour favoriser les décisions achat/vente
# Augmenter gamma pour punir plus les erreurs sur les classes directionnelles
# Utiliser alpha pour donner plus de poids aux classes vente (0) et achat (2)

# Calculer alpha pour favoriser vente et achat
directional_weight = (class_weights[0] + class_weights[2]) / 2
alpha_tensor = torch.tensor(
    [directional_weight, class_weights[1], directional_weight], device=device
)

best_metrics_focus = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    class_weights=class_weights,
    device=device,
    epochs=training_params["epochs"],
    lr=training_params["learning_rate"],
    grad_clip=training_params["grad_clip"],
    lambda_reg=training_params["lambda_reg"],
    lambda_vol=training_params["lambda_vol"],
    mixed_precision=training_params["mixed_precision"],
    output_dir=output_dir,
    save_every=training_params["save_every"],
    focal_gamma=3.0,  # Augmenter pour plus de focus sur les erreurs difficiles
    focal_alpha=alpha_tensor,  # Favoriser vente et achat
)
best_metrics_focus


In [ ]:
artifact_path = output_dir / "final_model_notebook.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": cfg.__dict__,
        "normalization": {
            "mean": stats.mean.tolist(),
            "std": stats.std.tolist(),
            "class_weights": stats.class_weights.tolist(),
            "tau_vocab_size": stats.tau_vocab_size,
            "reg_mean": stats.reg_mean,
            "reg_std": stats.reg_std,
            "vol_mean": stats.vol_mean,
            "vol_std": stats.vol_std,
            "vol_log": stats.vol_log,
        },
        "best_metrics": best_metrics,
        "params": training_params,
    },
    artifact_path,
)
print(f"Modèle sauvegardé dans: {artifact_path}")


In [ ]:
if best_metrics:
    metrics_df = pd.DataFrame(best_metrics, index=[0]).T.rename(columns={0: "valeur"})
    display(metrics_df)
    if "val_confusion" in best_metrics and best_metrics["val_confusion"]:
        confusion = np.array(best_metrics["val_confusion"])
        confusion_df = pd.DataFrame(
            confusion,
            index=["réel_-1", "réel_0", "réel_1"],
            columns=["prédit_-1", "prédit_0", "prédit_1"],
        )
        display(confusion_df)
else:
    print("Aucune métrique de validation enregistrée (best_metrics est vide).")


In [ ]:
## Évaluation avec seuil de confiance pour décisions plus tranchées

# Charger le modèle si nécessaire
if "model" not in globals():
    # Recharger depuis artifact_path si besoin
    checkpoint = torch.load(artifact_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])

model.eval()


# Fonction pour évaluer avec seuil
def evaluate_with_threshold(model, val_loader, device, confidence_threshold=0.6):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch in val_loader:
            X, y_cls, y_reg, y_vol, tau, y_bin = [x.to(device) for x in batch]
            outputs = model(X, tau)
            probs = torch.softmax(outputs, dim=1)
            max_probs, preds = torch.max(probs, dim=1)

            # Appliquer le seuil : si prob max < threshold, classer comme neutre (1)
            preds = torch.where(
                max_probs >= confidence_threshold, preds, torch.tensor(1, device=device)
            )

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_cls.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    from sklearn.metrics import confusion_matrix, classification_report

    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1, 2])
    report = classification_report(
        all_labels, all_preds, target_names=["vente", "neutre", "achat"]
    )

    print(f"Seuil de confiance: {confidence_threshold}")
    print("Matrice de confusion:")
    print(cm)
    print("\nRapport de classification:")
    print(report)

    return cm, report


# Tester avec différents seuils
thresholds = [0.5, 0.6, 0.7, 0.8]
for thresh in thresholds:
    print(f"\n{'=' * 50}")
    evaluate_with_threshold(model, val_loader, device, thresh)


## Prochaines étapes

- Ajustez les hyperparamètres ou les préfixes de jeu de données pour entraîner des variantes.
- Relancez l'entraînement en changeant `max_train_files` / `max_val_files` pour un smoke test rapide.
- Analysez les checkpoints générés dans `models/tft_notebook/` ou chargez-les pour de l'inférence.


## Améliorations pour des décisions plus tranchées sur achat/vente

### Modifications apportées :

1. **Évaluation avec seuil de confiance** : Nouvelle cellule qui applique un seuil minimum sur les probabilités prédites. Si la probabilité maximale est inférieure au seuil (ex. 0.6), la prédiction est forcée à "neutre". Cela réduit les faux positifs achat/vente et augmente la précision sur ces classes.

2. **Entraînement avec FocalLoss ajustée** : Nouvelle cellule d'entraînement utilisant une FocalLoss avec `gamma=3.0` (plus de pénalité sur les erreurs difficiles) et `alpha` favorisant les classes vente et achat. Cela aide le modèle à mieux distinguer les signaux directionnels des neutres.

### Conseils pour une meilleure décision :

- **Seuil de confiance** : Testez différents seuils (0.5 à 0.8) et choisissez celui qui maximise la précision sur achat/vente tout en gardant un rappel acceptable.
- **Direction margin dans le dataset** : Assurez-vous que le dataset a été généré avec un `direction_margin` approprié (ex. 0.0025) pour filtrer les signaux ambigus dès la labellisation.
- **Équilibrage des classes** : Si les classes achat/vente sont sous-représentées, activez `balance_classes=True` dans `prepare_dataloader`.
- **Post-traitement** : En inférence, appliquez toujours un seuil de confiance pour éviter les trades incertains.

Ces changements devraient rendre le modèle plus conservateur sur les décisions achat/vente, privilégiant la qualité à la quantité.
